## Running eval on lm-harness bennchmarks

The purpose of this notebook is to run evaluate huggingface-compatible models on lm-harness benchmarks

>NOTE: The lm-harness API for running the evaluation can help push results down to wandb. Hence, you can pass wandb token via the terminal using:



```
wandb login
```

To do evals just scan the code blocks and run appropriately.

In [ ]:
# Let's install lm-eval
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
!cd lm-evaluation-harness
!pip install -e .

In [ ]:
# Install wandb extra for pushing results to Weights & Biases
!pip install lm_eval[wandb]

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 80.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 68.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 61.0 MB/s eta 0

## Usage
The goal here is to run evals on the following MASA paper benchmark tasks
```
TASKS = [
    "piqa",
    "hellaswag",
    "lambada_openai",
    "arc_easy",
    "arc_challenge",
    "sciq",
    "mmlu",
    "wikitext",
]
```


### Run model on a single model

In [ ]:
# MASA benchmark tasks
TASKS = [
    "piqa",
    "hellaswag",
    "lambada_openai",
    "arc_easy",
    "arc_challenge",
    "sciq",
    "mmlu",
    "wikitext",
]

import subprocess

# Pick one model checkpoint
ckpt = "kokolamba/SubspaceDecoder_mha"

# Convert list to comma-separated string
tasks_arg = ",".join(TASKS)
model_name = ckpt.split("/")[-1]

print(f"Running evaluation for {ckpt}...")

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={ckpt},trust_remote_code=True",
    "--tasks", tasks_arg,
    "--device", "cuda:0",
    "--batch_size", "auto:4",
    "--output_path", "results",
    "--wandb_args", f"project=subspace-decoder-lm-harness-results,name={model_name}",
    "--log_samples",
]

# Run the command
subprocess.run(cmd, check=True)



### Run on multiple models

In [ ]:
# Let's test things out with one model first
model_checkpoints = [
    "kokolamba/SubspaceDecoder_mla192-96-0",
    "kokolamba/SubspaceDecoder_mla192-96-192",
    "kokolamba/SubspaceDecoder_mla0-128-0",
    "kokolamba/SubspaceDecoder_mla0-96-192",
    "kokolamba/SubspaceDecoder_mla0-0-192",
    "kokolamba/SubspaceDecoder_mla0-0-0",
    "kokolamba/SubspaceDecoder_mla192-0-0",
    "kokolamba/SubspaceDecoder_mha"
]

# Convert list to comma-separated string
tasks_arg = ",".join(TASKS)

for ckpt in model_checkpoints:

    model_name = ckpt.split("/")[-1]
    print(f"Running evaluation for {model_name}...")
    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={ckpt},trust_remote_code=True",
        "--trust_remote_code",
        "--tasks", tasks_arg,
        "--device", "cuda:0",
        "--batch_size", "auto:4",
        "--output_path", "results",
        "--wandb_args", f"project=subspace-decoder-lm-harness-results,name={model_name}",
        "--log_samples"
    ]
    subprocess.run(cmd, check=True)

## Earlier rough drafts

Who knows, it might be useful if the code above is buggy. I expect things to work out fine for the most part.

Just leaving this for the first non-buggy run and then the code blocks below can be removed!

At the very least, we maybe want to leave information about other lm-harness benchmark tasks we may want to evaluate.

In [ ]:
# Use lm-eval to evaluate a model on HF on based on the instructions below
# ```

# All datasets are to be evaluated using zero-shot except `mmlu` and `gsm8k`.

# `mmlu` should use `5 few-shot`

# `gsm8k` should use `8 few-shot`

# You can run the evaluation using the command below(1st set of 10 tasks)
# TASKS = [
#     "wikitext",         # Perplexity on WikiText
#     "lambada_openai",   # Cloze/Prediction task
#     "hellaswag",        # Commonsense NLI
#     "piqa",             # Physical Interaction QA
#     "winogrande",       # Commonsense Reasoning (Winograd Schema)
#     "arc_easy",         # AI2 Reasoning Challenge (Easy)
#     "arc_challenge",    # AI2 Reasoning Challenge (Challenge)
#     "openbookqa",       # Open Book Question Answering
#     "mmlu",             # Massive Multitask Language Understanding
#     "gsm8k",            # Grade School Math
# ]
# ``

# New tasks to test on (2nd set of 10 tasks)
# SciQ, RACE, ReCORD, SST, MRPC, RTE, MultiNLI, WSC273, WiC.
# TASKS = [
#     "sciq",
#     "race",
#     "mastermind",
#     "swag",
#     "anli",
#     "xnli",
#     "wsc273",
#     "pubmedqa",
#     "mathqa",
#     "commonsense_qa"
# ]

# MASA benchmark tasks
TASKS = [
    "piqa",
    "hellaswag",
    "lambada_openai",
    "arc_easy",
    "arc_challenge",
    "sciq",
    "mmlu",
    "wikitext",
]



# Let's test things out with one model first
model_checkpoints = [
    "kokolamba/SubspaceDecoder_mla192-96-0",
    "kokolamba/SubspaceDecoder_mla192-96-192",
    "kokolamba/SubspaceDecoder_mla0-128-0",
    "kokolamba/SubspaceDecoder_mla0-96-192",
    "kokolamba/SubspaceDecoder_mla0-0-192",
    # "kokolamba/SubspaceDecoder_mla0-0-0",
    "kokolamba/SubspaceDecoder_mla192-0-0",
    "kokolamba/SubspaceDecoder_mha"
]

# !lm_eval --model hf \
#     --model_args pretrained=kokolamba/SubspaceDecoder_mha,trust_remote_code=True \
#     --tasks wikitext,lambada_openai,hellaswag,piqa,winograde,arc_easy,arc_challenge,openbookqa,mmlu,gsm8k_cot\
#     --device cuda:0 \
#     --batch_size auto:4 \
#     --output_path results \
#     --wandb_args project=subspace-decoder-lm-harness-results \
#     --log_samples \
#     --limit 10

# Pick one model checkpoint
# ckpt = "kokolamba/SubspaceDecoder_mha"

# print(f"Running evaluation for {ckpt}...")

# cmd = [
#     "lm_eval",
#     "--model", "hf",
#     "--model_args", f"pretrained={ckpt},trust_remote_code=True",
#     "--tasks", "wikitext,lambada_openai,hellaswag,piqa,winogrande,arc_easy,arc_challenge,openbookqa,mmlu,gsm8k_cot",
#     "--device", "cuda:0",
#     "--batch_size", "auto:4",
#     "--output_path", "results",
#     "--wandb_args", "project=subspace-decoder-lm-harness-results",
#     "--log_samples",
#     "--limit", "10"
# ]

# # Run the command
# subprocess.run(cmd, check=True)

In [14]:
#!lm-eval --tasks list_groups

It's now gonna be remaining just `kokolamba/SubspaceDecoder_mla0-0-0` with best checkpoint `checkpoint-3000` as subfolder.

In [ ]:
# Use subprocess to run the command for each model in the list
import subprocess

# Convert list to comma-separated string
tasks_arg = ",".join(TASKS)

for ckpt in model_checkpoints:
    print(f"Running evaluation for {ckpt}...")
    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={ckpt},trust_remote_code=True",
        "--trust_remote_code",
        "--tasks", tasks_arg,
        "--device", "cuda:0",
        "--batch_size", "auto:4",
        "--output_path", "results",
        "--wandb_args", "project=subspace-decoder-lm-harness-results",
        "--log_samples"
    ]
    subprocess.run(cmd, check=True)

Running evaluation for kokolamba/SubspaceDecoder_mla192-96-0...


wandb: Currently logged in as: abdulhakeemadefioye (abdulhakeemadefioye-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.22.1
wandb: Run data is saved locally in /workspace/wandb/run-20250930_163603-gh6dk07t
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run trim-water-17
wandb: ⭐️ View project at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results
wandb: 🚀 View run at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results/runs/gh6dk07t
2025-09-30:16:36:06 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-30:16:36:06 INFO     [__main__:429] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-09-30:16:36:06 INFO     [__main__:446] Selected Tasks: ['anli', 'commonsense_qa', 'mastermind', 'mathqa', 'pubmedqa', 'race', 'sciq', '

config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192
config.q_shared_dim 192


Generating validation split: 100%|██████████| 4475/4475 [00:00<00:00, 44995.06 examples/s]
/usr/local/lib/python3.11/dist-packages/datasets/load.py:1231: FutureWarning: The repository for bigbio/pubmed_qa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/pubmed_qa
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
Generating train split: 450 examples [00:00, 20656.85 examples/s]
Generating validation split: 50 examples [00:00, 12357.27 examples/s]
Generating test split: 500 examples [00:00, 20750.12 examples/s]
Generating test split: 100%|██████████| 20005/20005 [00:00<00:00, 816264.10 examples/s]
/usr/local/lib/python3.11/dist-packages/datasets/load.py:1231: FutureWarning: The repository for winograd_wsc contains custom code

Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 32
Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 64
hf (pretrained=kokolamba/SubspaceDecoder_mla192-96-0,trust_remote_code=True,trust_remote_code=True), gen_kwargs: (None), limit: 10.0, num_fewshot: None, batch_size: auto:4 (32,64,64,64,64)
|      Tasks       |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|------------------|-------|------|-----:|--------|---|-----:|---|-----:|
|anli_r1           |      1|none  |     0|acc     |↑  |0.3000|±  |0.1528|
|anli_r2           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|anli_r3           |      1|none  |     0|acc     |↑  |0.6000|±  |0.1633|
|commonsense_qa    |Yaml   |none  |     0|acc     |↑  |0.5000|±  |0.1667|
|mastermind_24_easy|      1|none  |     0|acc     |↑  |0.1000|±  |0.1000|
|mastermind_24_hard|      1|none  |     0|acc     |↑  |0.1000|±  |0.1000|
|mastermind_35_

...

Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 32
Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 64
hf (pretrained=kokolamba/SubspaceDecoder_mla192-0-0,trust_remote_code=True,trust_remote_code=True), gen_kwargs: (None), limit: 10.0, num_fewshot: None, batch_size: auto:4 (32,64,64,64,64)
|      Tasks       |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|------------------|-------|------|-----:|--------|---|-----:|---|-----:|
|anli_r1           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|anli_r2           |      1|none  |     0|acc     |↑  |0.3000|±  |0.1528|
|anli_r3           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|commonsense_qa    |Yaml   |none  |     0|acc     |↑  |0.5000|±  |0.1667|
|mastermind_24_easy|      1|none  |     0|acc     |↑  |0.1000|±  |0.1000|
|mastermind_24_hard|      1|none  |     0|acc     |↑  |0.1000|±  |0.1000|
|mastermind_35_e

wandb: Currently logged in as: abdulhakeemadefioye (abdulhakeemadefioye-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.22.1
wandb: Run data is saved locally in /workspace/wandb/run-20250930_165317-tedcd8x4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run morning-water-23
wandb: ⭐️ View project at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results
wandb: 🚀 View run at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results/runs/tedcd8x4
2025-09-30:16:53:21 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-30:16:53:21 INFO     [__main__:429] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-09-30:16:53:21 INFO     [__main__:446] Selected Tasks: ['anli', 'commonsense_qa', 'mastermind', 'mathqa', 'pubmedqa', 'race', 'sciq'

config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.


100%|██████████| 10/10 [00:00<00:00, 1899.16it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for mastermind_46_easy on rank 0...
100%|██████████| 10/10 [00:00<00:00, 1854.82it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for mastermind_46_hard on rank 0...
100%|██████████| 10/10 [00:00<00:00, 1833.82it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for commonsense_qa on rank 0...
100%|██████████| 10/10 [00:00<00:00, 845.98it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for anli_r1 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 2968.58it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for anli_r2 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 3092.46it/s]
2025-09-30:16:55:37 INFO     [api.task:434] Building contexts for anli_r3 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 3114.04it/s]
2025-09-30:16:55:38 INFO     [evaluator:574] Running loglikelihood requests
Running loglikelihood re

Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 32
Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 64
hf (pretrained=kokolamba/SubspaceDecoder_mha,trust_remote_code=True,trust_remote_code=True), gen_kwargs: (None), limit: 10.0, num_fewshot: None, batch_size: auto:4 (32,64,64,64,64)
|      Tasks       |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|------------------|-------|------|-----:|--------|---|-----:|---|-----:|
|anli_r1           |      1|none  |     0|acc     |↑  |0.5000|±  |0.1667|
|anli_r2           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|anli_r3           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|commonsense_qa    |Yaml   |none  |     0|acc     |↑  |0.5000|±  |0.1667|
|mastermind_24_easy|      1|none  |     0|acc     |↑  |0.0000|±  |0.0000|
|mastermind_24_hard|      1|none  |     0|acc     |↑  |0.1000|±  |0.1000|
|mastermind_35_easy|   

In [21]:
tasks_arg

'sciq,race,mastermind,swag,anli,xnli,wsc273,pubmedqa,mathqa,commonsense_qa'

In [22]:
# Lets run checkpoint with subfolder
# Pick one model checkpoint
ckpt = "kokolamba/SubspaceDecoder_mla0-0-0"

print(f"Running evaluation for {ckpt}...")

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={ckpt},trust_remote_code=True,subfolder=checkpoint-3000",
    "--tasks", tasks_arg,
    "--trust_remote_code",
    "--device", "cuda:0",
    "--batch_size", "auto:4",
    "--output_path", "results",
    "--wandb_args", "project=subspace-decoder-lm-harness-results",
    "--log_samples",
    "--limit", "10"
]

# Run the command
subprocess.run(cmd, check=True)

Running evaluation for kokolamba/SubspaceDecoder_mla0-0-0...


wandb: Currently logged in as: abdulhakeemadefioye (abdulhakeemadefioye-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.22.1
wandb: Run data is saved locally in /workspace/wandb/run-20250930_170143-6h2mu97j
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run hearty-shape-24
wandb: ⭐️ View project at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results
wandb: 🚀 View run at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results/runs/6h2mu97j
2025-09-30:17:01:47 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-30:17:01:47 INFO     [__main__:429] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-09-30:17:01:47 INFO     [__main__:446] Selected Tasks: ['anli', 'commonsense_qa', 'mastermind', 'mathqa', 'pubmedqa', 'race', 'sciq',

config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.
config.q_shared_dim None
Using identity for shared projection.


100%|██████████| 10/10 [00:00<00:00, 1822.26it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for mastermind_46_easy on rank 0...
100%|██████████| 10/10 [00:00<00:00, 1824.80it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for mastermind_46_hard on rank 0...
100%|██████████| 10/10 [00:00<00:00, 1800.29it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for commonsense_qa on rank 0...
100%|██████████| 10/10 [00:00<00:00, 831.71it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for anli_r1 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 2927.14it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for anli_r2 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 3058.41it/s]
2025-09-30:17:04:04 INFO     [api.task:434] Building contexts for anli_r3 on rank 0...
100%|██████████| 10/10 [00:00<00:00, 3060.20it/s]
2025-09-30:17:04:04 INFO     [evaluator:574] Running loglikelihood requests
Running loglikelihood re

Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 32
Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 64
hf (pretrained=kokolamba/SubspaceDecoder_mla0-0-0,trust_remote_code=True,subfolder=checkpoint-3000,trust_remote_code=True), gen_kwargs: (None), limit: 10.0, num_fewshot: None, batch_size: auto:4 (32,64,64,64,64)
|      Tasks       |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|------------------|-------|------|-----:|--------|---|-----:|---|-----:|
|anli_r1           |      1|none  |     0|acc     |↑  |0.5000|±  |0.1667|
|anli_r2           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|anli_r3           |      1|none  |     0|acc     |↑  |0.4000|±  |0.1633|
|commonsense_qa    |Yaml   |none  |     0|acc     |↑  |0.5000|±  |0.1667|
|mastermind_24_easy|      1|none  |     0|acc     |↑  |0.0000|±  |0.0000|
|mastermind_24_hard|      1|none  |     0|acc     |↑  |0.1000|±  |

CompletedProcess(args=['lm_eval', '--model', 'hf', '--model_args', 'pretrained=kokolamba/SubspaceDecoder_mla0-0-0,trust_remote_code=True,subfolder=checkpoint-3000', '--tasks', 'sciq,race,mastermind,swag,anli,xnli,wsc273,pubmedqa,mathqa,commonsense_qa', '--trust_remote_code', '--device', 'cuda:0', '--batch_size', 'auto:4', '--output_path', 'results', '--wandb_args', 'project=subspace-decoder-lm-harness-results', '--log_samples', '--limit', '10'], returncode=0)

## Let's now run on GPT2

In [24]:
# Lets run checkpoint with subfolder
# Pick one model checkpoint
ckpt = "openai-community/gpt2"

# You can run the evaluation using the command below(1st set of 10 tasks)
TASKS = [
    "wikitext",         # Perplexity on WikiText
    "lambada_openai",   # Cloze/Prediction task
    "hellaswag",        # Commonsense NLI
    "piqa",             # Physical Interaction QA
    "winogrande",       # Commonsense Reasoning (Winograd Schema)
    "arc_easy",         # AI2 Reasoning Challenge (Easy)
    "arc_challenge",    # AI2 Reasoning Challenge (Challenge)
    "openbookqa",       # Open Book Question Answering
    "mmlu",             # Massive Multitask Language Understanding
    "gsm8k",            # Grade School Math
]

# New tasks to test on (2nd set of 10 tasks)
# SciQ, RACE, ReCORD, SST, MRPC, RTE, MultiNLI, WSC273, WiC.
# TASKS = [
#     "sciq",
#     "race",
#     "mastermind",
#     "swag",
#     "anli",
#     "xnli",
#     "wsc273",
#     "pubmedqa",
#     "mathqa",
#     "commonsense_qa"
# ]

tasks_arg = ",".join(TASKS)

print(f"Running evaluation for {ckpt}...")

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={ckpt},trust_remote_code=True",
    "--tasks", tasks_arg,
    "--trust_remote_code",
    "--device", "cuda:0",
    "--batch_size", "auto:4",
    "--output_path", "results",
    "--wandb_args", "project=subspace-decoder-lm-harness-results",
    "--log_samples",
    "--limit", "10"
]

# Run the command
subprocess.run(cmd, check=True)

Running evaluation for openai-community/gpt2...


wandb: Currently logged in as: abdulhakeemadefioye (abdulhakeemadefioye-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.22.1
wandb: Run data is saved locally in /workspace/wandb/run-20250930_171348-shc4diob
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run wandering-sea-26
wandb: ⭐️ View project at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results
wandb: 🚀 View run at https://wandb.ai/abdulhakeemadefioye-personal/subspace-decoder-lm-harness-results/runs/shc4diob
2025-09-30:17:13:51 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-30:17:13:51 INFO     [__main__:429] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-09-30:17:13:51 INFO     [__main__:446] Selected Tasks: ['arc_challenge', 'arc_easy', 'gsm8k', 'hellaswag', 'lambada_openai', 'mmlu',

  2025-09-30T17:15:35.372761Z  WARN  Status Code: 504. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-30T17:15:35.372797Z  WARN  Retry attempt #0. Sleeping 1.223136662s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



Generating test split: 100%|██████████| 3084/3084 [00:00<00:00, 450830.67 examples/s]
2025-09-30:17:15:39 WARNING  [api.task:846] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2025-09-30:17:15:39 WARNING  [api.task:858] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2025-09-30:17:15:39 WARNING  [api.task:846] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2025-09-30:17:15:39 WARNING  [api.task:858] [Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2025-09-30:17:15:39 WARNING  [api.task:846] [Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
2025-09-30:17:15:39 WARNING  [api.task:858] [Task: wikitext] metric bits_per_byte is defined, but hi

Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 32
Passed argument batch_size = auto:4.0. Detecting largest batch size
Determined largest batch size: 64
Passed argument batch_size = auto. Detecting largest batch size
Determined Largest batch size: 32


...

100%|██████████| 100/100 [00:00<00:00, 7160.33it/s]
fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
wandb: WARNING Serializing object of type str that is 105896 bytes
wandb: WARNING Serializing object of type str that is 105896 bytes
wandb: WARNING Serializing object of type str that is 112550 bytes
wandb: WARNING Serializing object of type str that is 112550 bytes
2025-09-30:17:17:42 INFO     [loggers.evaluation_tracker:209] Saving results aggregated
2025-09-30:17:17:42 INFO     [loggers.evaluation_tracker:298] Saving per-sample results for: arc_challenge
2025-09-30:17:17:42 INFO     [loggers.evaluation_tracker:298] Saving per-sample results for: arc_easy
2025-09-30:17:17:42 INFO     [loggers.evaluation_tracker:298] Saving per-sample results for: gsm8k
2025-09-30:17:17:42 INFO     [loggers.evaluation_tracker:298] Saving per-sample results for: hellaswag
2025-09-30:17:17:42 INFO     [loggers.eval

hf (pretrained=openai-community/gpt2,trust_remote_code=True,trust_remote_code=True), gen_kwargs: (None), limit: 10.0, num_fewshot: None, batch_size: auto:4 (32,64,64,64,64)
|                 Tasks                 |Version|     Filter     |n-shot|    Metric     |   | Value |   | Stderr |
|---------------------------------------|------:|----------------|-----:|---------------|---|------:|---|--------|
|arc_challenge                          |      1|none            |     0|acc            |↑  | 0.1000|±  |  0.1000|
|                                       |       |none            |     0|acc_norm       |↑  | 0.1000|±  |  0.1000|
|arc_easy                               |      1|none            |     0|acc            |↑  | 0.3000|±  |  0.1528|
|                                       |       |none            |     0|acc_norm       |↑  | 0.1000|±  |  0.1000|
|gsm8k                                  |      3|flexible-extract|     5|exact_match    |↑  | 0.1000|±  |  0.1000|
|                     

wandb: uploading artifact run-shc4diob-winogrande_eval_results; uploading artifact winogrande; updating run metadata
wandb: 
wandb: Run history:
wandb:                  arc_challenge/acc ▁
wandb:             arc_challenge/acc_norm ▁
wandb:      arc_challenge/acc_norm_stderr ▁
wandb:           arc_challenge/acc_stderr ▁
wandb:                       arc_easy/acc ▁
wandb:                  arc_easy/acc_norm ▁
wandb:           arc_easy/acc_norm_stderr ▁
wandb:                arc_easy/acc_stderr ▁
wandb: gsm8k/exact_match,flexible-extract ▁
wandb:     gsm8k/exact_match,strict-match ▁
wandb:                               +147 ...
wandb: 
wandb: Run summary:
wandb:             arc_challenge/acc 0.1
wandb:        arc_challenge/acc_norm 0.1
wandb: arc_challenge/acc_norm_stderr 0.1
wandb:      arc_challenge/acc_stderr 0.1
wandb:           arc_challenge/alias arc_challenge
wandb:                  arc_easy/acc 0.3
wandb:             arc_easy/acc_norm 0.1
wandb:      arc_easy/acc_norm_stderr 0.1
wan


|      Groups      |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|------------------|------:|------|------|------|---|-----:|---|-----:|
|mmlu              |      2|none  |      |acc   |↑  |0.2579|±  |0.0184|
| - humanities     |      2|none  |      |acc   |↑  |0.2538|±  |0.0391|
| - other          |      2|none  |      |acc   |↑  |0.2462|±  |0.0377|
| - social sciences|      2|none  |      |acc   |↑  |0.2917|±  |0.0417|
| - stem           |      2|none  |      |acc   |↑  |0.2474|±  |0.0313|



CompletedProcess(args=['lm_eval', '--model', 'hf', '--model_args', 'pretrained=openai-community/gpt2,trust_remote_code=True', '--tasks', 'wikitext,lambada_openai,hellaswag,piqa,winogrande,arc_easy,arc_challenge,openbookqa,mmlu,gsm8k', '--trust_remote_code', '--device', 'cuda:0', '--batch_size', 'auto:4', '--output_path', 'results', '--wandb_args', 'project=subspace-decoder-lm-harness-results', '--log_samples', '--limit', '10'], returncode=0)